In [1]:
import pandas as pd
import os

# بارگذاری فایل CSV
file_path = 'hbond_extra_mix_res_run3_amino.csv'  # مسیر فایل CSV خود را مشخص کنید
data = pd.read_csv(file_path)

# ایجاد پوشه برای ذخیره نتایج
output_folder = r'C:\Users\Lenovo\Desktop\molecular bonds _mix\output_folder'
os.makedirs(output_folder, exist_ok=True)


# شناسایی ترکیبهای منحصر به فرد از Residue1 و Residue3
unique_combinations = data[['Residue1Amino', 'Residue3Amino']].drop_duplicates()

# مجموعهای برای پیگیری ترکیبهای ذخیره شده
saved_combinations = set()

# ذخیرهسازی دادهها بر اساس ترکیبهای منحصر به فرد
for index, row in unique_combinations.iterrows():
    residue1 = row['Residue1Amino']
    residue3 = row['Residue3Amino']
    
    # ایجاد یک کلید برای ترکیب (به صورت ترتیبی)
    combination_key = tuple(sorted([residue1, residue3]))

    # بررسی اینکه آیا این ترکیب قبلاً ذخیره شده است یا خیر
    if combination_key not in saved_combinations:
        # فیلتر کردن دادهها برای ترکیب خاص و معکوس
        filtered_data = data[((data['Residue1Amino'] == residue1) & (data['Residue3Amino'] == residue3)) |
                             ((data['Residue1Amino'] == residue3) & (data['Residue3Amino'] == residue1))]
        
        # ساخت نام فایل برای ذخیرهسازی
        output_file_name = f"{residue1}_{residue3}.csv"
        output_file_path = os.path.join(output_folder, output_file_name)
        
        # ذخیرهسازی دادههای فیلتر شده در یک فایل CSV جدید
        filtered_data.to_csv(output_file_path, index=False)
        
        # اضافه کردن ترکیب به مجموعه ذخیره شده
        saved_combinations.add(combination_key)

print(f"Data saved to {output_folder}")

Data saved to C:\Users\Lenovo\Desktop\molecular bonds _mix\output_folder


In [2]:
import pandas as pd
import os

# مسیر پوشه حاوی فایلهای CSV
input_folder = r'C:\Users\Lenovo\Desktop\molecular bonds _mix\output_folder'  # مسیر پوشه ورودی
output_folder = r'C:\Users\Lenovo\Desktop\molecular bonds _mix\output_folder_count'  # مسیر پوشه خروجی
os.makedirs(output_folder, exist_ok=True)

# لیست کردن همه فایلهای CSV در پوشه ورودی
csv_files = [f for f in os.listdir(input_folder) if f.endswith('.csv')]

for file_name in csv_files:
    file_path = os.path.join(input_folder, file_name)
    
    # بارگذاری دادهها از فایل CSV
    data = pd.read_csv(file_path)
    
    # بررسی وجود ستون 'Frame'
    if 'Frame' not in data.columns:
        print(f"Column 'Frame' not found in {file_name}. Skipping this file.")
        continue

    # محاسبه تعداد تکرارهای فریم
    data['count'] = (data['Frame'] != data['Frame'].shift()).cumsum()
    counts = data.groupby('count').agg(Frame=('Frame', 'first')).reset_index(drop=True)
    
    # اضافه کردن ستون count به دادهها
    data['count'] = data.groupby('count')['Frame'].transform('size')

    # حذف ردیفهای تکراری بر اساس فریم و شمارنده
    unique_data = data.drop_duplicates(subset=['Frame'])

    # ساخت نام فایل جدید برای ذخیرهسازی
    output_file_path = os.path.join(output_folder, file_name)

    # ذخیرهسازی دادههای جدید در فایل CSV
    unique_data.to_csv(output_file_path, index=False)

print(f"Processed files saved to {output_folder}")

Processed files saved to C:\Users\Lenovo\Desktop\molecular bonds _mix\output_folder_count


In [3]:
import os
import pandas as pd

# مسیرهای ورودی و خروجی
input_folder = r'C:\Users\Lenovo\Desktop\molecular bonds _mix\output_folder_count'
output_folder = r'C:\Users\Lenovo\Desktop\molecular bonds _mix\output_folder_final'

# ایجاد پوشه خروجی اگر وجود ندارد
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

# لیست فایل‌های CSV موجود در پوشه ورودی
csv_files = [f for f in os.listdir(input_folder) if f.endswith('.csv')]

# پردازش هر فایل CSV
for file in csv_files:
    file_path = os.path.join(input_folder, file)
    df = pd.read_csv(file_path)

    # مرتب‌سازی داده‌ها بر اساس ستون 'Frame'
    df = df.sort_values(by='Frame')

    # لیست نتایج
    results = []

    # بررسی فریم‌های متوالی
    current_sequence = []
    for index, row in df.iterrows():
        if current_sequence and row['Frame'] != current_sequence[-1]['Frame'] + 1:
            if len(current_sequence) >= 3:
                first_frame = current_sequence[0]['Frame']
                last_frame = current_sequence[-1]['Frame']
                amino_1 = current_sequence[0]['Residue1Amino']
                amino_3 = current_sequence[-1]['Residue3Amino']
                total_frames = len(current_sequence)
                total_count = sum([r['count'] for r in current_sequence])
                results.append([amino_1, amino_3, first_frame, last_frame, total_frames, total_count])
            current_sequence = []
        current_sequence.append(row)

    # آخرین توالی متوالی را بررسی می‌کنیم
    if len(current_sequence) >= 3:
        first_frame = current_sequence[0]['Frame']
        last_frame = current_sequence[-1]['Frame']
        amino_1 = current_sequence[0]['Residue1Amino']
        amino_3 = current_sequence[-1]['Residue3Amino']
        total_frames = len(current_sequence)
        total_count = sum([r['count'] for r in current_sequence])
        results.append([amino_1, amino_3, first_frame, last_frame, total_frames, total_count])

    # تبدیل نتایج به DataFrame جدید و ذخیره آن به عنوان فایل CSV جداگانه
    results_df = pd.DataFrame(results, columns=['Amino 1', 'Amino 3', 'first frame', 'final frame', 'total frame count', 'total hbond count'])
    output_file_path = os.path.join(output_folder, f'processed_{file}')
    results_df.to_csv(output_file_path, index=False)

# نمایش پیغام تکمیل پردازش
print('پردازش تمام فایل‌ها به پایان رسید.')


پردازش تمام فایل‌ها به پایان رسید.


In [4]:
import pandas as pd
import os

# تعریف مسیر پوشه حاوی فایلهای CSV
input_folder =  r'C:\Users\Lenovo\Desktop\molecular bonds _mix\output_folder_final'
# تعریف نام فایل خروجی
output_file = r'C:\Users\Lenovo\Desktop\molecular bonds _mix\combined_output.csv'


# لیست فایلهای CSV در پوشه ورودی
csv_files = [f for f in os.listdir(input_folder) if f.endswith('.csv')]

# لیست برای ذخیره DataFrame ها
dataframes = []

# پردازش هر فایل CSV و افزودن آن به لیست DataFrame ها
for csv_file in csv_files:
    file_path = os.path.join(input_folder, csv_file)
    
    # بررسی اینکه آیا فایل خالی است
    if os.path.getsize(file_path) > 0:
        try:
            df = pd.read_csv(file_path)
            
            # بررسی اینکه آیا DataFrame دارای ستون است یا خیر
            if not df.empty and not df.columns.empty:
                dataframes.append(df)
        except Exception as e:
            print(f"خطا در پردازش فایل {csv_file}: {e}")

# بررسی اینکه آیا هیچ DataFrame ای وجود دارد
if dataframes:
    # ترکیب تمامی DataFrame ها به یک DataFrame واحد
    combined_df = pd.concat(dataframes, ignore_index=True)

    # ذخیره DataFrame ترکیبی به یک فایل CSV جدید
    combined_df.to_csv(output_file, index=False)
    print("تمامی فایلهای دارای داده به یک فایل CSV جدید ترکیب شدند و در مسیر مشخص شده ذخیره شدند.")
else:
    print("هیچ فایلی با دادههای معتبر پیدا نشد.")

تمامی فایلهای دارای داده به یک فایل CSV جدید ترکیب شدند و در مسیر مشخص شده ذخیره شدند.


In [5]:
import pandas as pd
# نام فایل CSV ورودی
input_file = r'C:\Users\Lenovo\Desktop\molecular bonds _mix\combined_output.csv'  # نام فایل خود را وارد کنید

# خواندن فایل CSV به عنوان DataFrame
data = pd.read_csv(input_file) 

# مرتبسازی دادهها بر اساس ستون 'first frame'
data_sorted = data.sort_values(by='first frame')

# حذف سطرهای تکراری
data_unique = data_sorted.drop_duplicates()

# ذخیره نتیجه به یک فایل CSV جدید
data_unique.to_csv(r'C:\Users\Lenovo\Desktop\molecular bonds _mix\combined_mix_res_run3.csv', index=False)
print("ستونهای تکراری حذف شدند و دادهها در فایل جدید ذخیره شدند.")

ستونهای تکراری حذف شدند و دادهها در فایل جدید ذخیره شدند.
